In [1]:
%pip install torch transformers peft accelerate datasets pandas tqdm tabulate sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 36.2 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 69.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.9/798.9 kB 103.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 196.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 86.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 70.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 169.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 153.2 MB/s eta 0:00:00

[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated package

In [5]:
# -*- coding: utf-8 -*-
"""
정성 평가 스크립트 (모델 '자체 지식력' 비교 전용)
- test02.jsonl에서 question과 reference answer만 사용 (✅ contexts 제공 안 함)
- 동일 시스템 프롬프트 + 재현성(do_sample=False, temperature=0.0)
- Base / Finetuned 답변 생성 후, Reference Answer와 함께 CSV 저장
- 사람이 직접 점수 기입할 수 있도록 정성 지표 열(accuracy/completeness/clarity/overall/rater/notes) 포함
- CSV 상단에 열 설명을 주석(#)으로 삽입
"""

import json
import random
import torch
import pandas as pd
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# =========================
# 경로/설정
# =========================
BASE_MODEL_PATH = "kakaocorp/kanana-1.5-8b-instruct-2505"
ADAPTER_PATH    = "ki-student/kanana-finetuned-model-v1"
TEST_DATA_PATH  = "./test02.jsonl"

OUTPUT_CSV_PATH = "qualitative_comparison_kanana03.csv"   # 산출물 경로
MAX_SAMPLES     = None  # None이면 전체, 숫자 지정 시 앞에서부터 제한 (예: 200)

# 모델 '자체 지식력'을 평가하므로, 문맥을 주지 않는 간단한 프롬프트 사용
SYSTEM_PROMPT = (
    "당신은 자동차 도메인에 정통한 '자동차 전문 AI'입니다. "
    "아는 사실만으로 간결하고 정확하게 답변하세요. "
    "모호하면 추정하지 말고 모른다고 말하세요."
)

# 재현성
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

GEN_KW = dict(
    max_new_tokens=256,
    do_sample=False,      # 재현성
    temperature=0.0,      # 재현성
    top_p=1.0,
    repetition_penalty=1.05,
)

# =========================
# 데이터 로드
# =========================
def load_test_jsonl(path: str):
    with open(path, "r", encoding="utf-8") as f:
        data = [json.loads(line) for line in f if line.strip()]
    # 필수 필드만 유지
    cleaned = []
    for it in data:
        q = it.get("question")
        a = it.get("answer")
        if q and a:
            cleaned.append({"question": q, "answer": a})
    return cleaned

# =========================
# 모델/생성
# =========================
def load_model(model_path, adapter_path=None):
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        trust_remote_code=True,
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    )
    tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    if adapter_path:
        model = PeftModel.from_pretrained(model, adapter_path).merge_and_unload()
    return model, tokenizer

@torch.no_grad()
def generate_answer(model, tokenizer, question: str) -> str:
    # ✅ contexts 없이, system + user만 구성
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    prompt_str = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_str, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        eos_token_id=tokenizer.eos_token_id,
        **GEN_KW
    )
    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # 안전 파싱: 프롬프트 길이만큼 잘라 잔여를 답변으로
    prompt_only = tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)
    answer = decoded[len(prompt_only):].strip()
    return answer

# =========================
# 메인
# =========================
def main():
    # 1) 데이터 로드
    raw = load_test_jsonl(TEST_DATA_PATH)
    if MAX_SAMPLES:
        raw = raw[:MAX_SAMPLES]
    print(f"✅ 로드된 샘플 수: {len(raw)}")

    # 2) 모델 로드
    base_model, tok = load_model(BASE_MODEL_PATH)
    ft_model, _     = load_model(BASE_MODEL_PATH, ADAPTER_PATH)

    # 3) 생성
    questions, refs = [], []
    base_answers, ft_answers = [], []

    print("\n--- 베이스 모델 생성 ---")
    for item in tqdm(raw, desc="Base"):
        q = item["question"]
        a = item["answer"]
        ans_base = generate_answer(base_model, tok, q)

        questions.append(q)
        refs.append(a)
        base_answers.append(ans_base)

    print("\n--- 파인튜닝 모델 생성 ---")
    for q in tqdm(questions, desc="Finetuned"):  # 위에서 필터된 질문 수와 동일하게
        ans_ft = generate_answer(ft_model, tok, q)
        ft_answers.append(ans_ft)

    # 4) CSV 구성 (정성 평가용 점수 칼럼 포함)
    df = pd.DataFrame({
        "Question": questions,
        "Reference Answer": refs,    # 평가자가 참조할 기준 정답
        "Base_Answer": base_answers,
        "Finetuned_Answer": ft_answers,

        # 정성 평가(사람이 기입)
        "accuracy": "",      # 정답과의 정확성(1~5)
        "completeness": "",  # 정보의 충분성/누락 없음(1~5)
        "clarity": "",       # 표현의 명료성/가독성(1~5)
        "overall": "",       # 종합(1~5)
        "rater": "",         # 평가자 ID/이름
        "notes": "",         # 코멘트
    })

    # 5) CSV 저장 (헤더 주석 포함)
    with open(OUTPUT_CSV_PATH, "w", encoding="utf-8-sig") as f:
        f.write("# Question: 평가 질문 (test02.jsonl의 question)\n")
        f.write("# Reference Answer: 기준 정답 (test02.jsonl의 answer)\n")
        f.write("# Base_Answer: 베이스 모델의 자체 지식 기반 답변 (contexts 미제공)\n")
        f.write("# Finetuned_Answer: 파인튜닝 모델의 자체 지식 기반 답변 (contexts 미제공)\n")
        f.write("# accuracy: 정답과의 정확성 (1~5)\n")
        f.write("# completeness: 정보의 충분성/누락 없음 (1~5)\n")
        f.write("# clarity: 표현의 명료성/가독성 (1~5)\n")
        f.write("# overall: 종합 점수 (1~5)\n")
        f.write("# rater: 평가자 이름/ID\n")
        f.write("# notes: 자유 코멘트\n")
        df.to_csv(f, index=False)

    print(f"\n✅ CSV 저장 완료: {OUTPUT_CSV_PATH}")
    print("👉 이제 이 CSV에서 각 문항별로 accuracy/completeness/clarity/overall 점수를 기입해 정성 평가를 진행하세요.")

if __name__ == "__main__":
    main()


✅ 로드된 샘플 수: 289


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


--- 베이스 모델 생성 ---


Base: 100%|██████████| 289/289 [15:09<00:00,  3.15s/it]



--- 파인튜닝 모델 생성 ---


Finetuned:  72%|███████▏  | 208/289 [04:01<01:52,  1.39s/it]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Finetuned: 100%|██████████| 289/289 [05:40<00:00,  1.18s/it]


✅ CSV 저장 완료: qualitative_comparison_kanana03.csv
👉 이제 이 CSV에서 각 문항별로 accuracy/completeness/clarity/overall 점수를 기입해 정성 평가를 진행하세요.


In [3]:
# -*- coding: utf-8 -*-
"""
정량 평가 스크립트 (CSV 전용, 지표 = Cosine Similarity / CtxAcc)
- 모델 출력 JSON에서 answer / context number 분리
- Cosine Similarity는 answer만으로 계산 (문장 임베딩 기반)
- CtxAcc(문맥 정확도)는 context number 정확도
- 결과는 CSV 한 파일로 저장
"""

import re
import json
import unicodedata
from typing import List, Dict, Any, Tuple

import torch
import torch.nn.functional as F
import pandas as pd
from tqdm import tqdm
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# -----------------------------
# 경로/모델 설정
# -----------------------------
BASE_MODEL_PATH = "kakaocorp/kanana-1.5-8b-instruct-2505"
ADAPTER_PATH    = "ki-student/kanana-finetuned-model-v1"
TEST_DATA_PATH  = "./test02.jsonl"
OUTPUT_CSV_PATH = "./quantitative_evaluation_kanana03.csv"

# 문장 임베딩 모델(코사인 유사도 계산용)
EMB_MODEL_NAME  = "BAAI/bge-m3"

# 훈련 시 시스템 프롬프트와 동일
SYSTEM_PROMPT = (
    "당신은 자동차 디자인 트렌드와 역사에 정통한 '자동차 디자인 전문 AI'입니다. "
    "특히 현대자동차의 디자인 철학인 '센슈어스 스포티니스'와 '플루이딕 스컬프처'를 깊이 이해하고 있습니다. "
    "사용자의 질문에 대해, 제공된 문맥을 참고하여 전문 지식을 바탕으로 상세하게 설명해주세요. "
    "답변은 반드시 아래 예시와 같이 JSON 형식으로 생성해야 하며, 어떤 문맥을 참고했는지 `context number` 필드에 해당 인덱스를 '[숫자]' 형식으로 포함해야 합니다."
    "\n\n"
    "답변 예시: {\"context number\": \"[1]\", \"answer\": \"플루이딕 스컬프처는 물이나 바람이 흐르는 듯한 유기적인 선을 강조하는 디자인 철학입니다.\"}"
)

# -----------------------------
# 유틸: JSON 안전 파싱 / 정규화 / context index
# -----------------------------
JSON_OBJ_RE = re.compile(r"\{.*?\}", flags=re.S)
CTX_NUM_RE  = re.compile(r'\[\s*(\d+)\s*\]')

def safe_json_answer(text: str) -> Tuple[str, str]:
    """
    모델 출력에서 마지막 JSON 객체를 찾아 answer와 context number를 분리 추출.
    실패 시: answer는 전체 텍스트, context는 텍스트에서 [i] 패턴 탐색.
    """
    answer = text.strip()
    ctx_raw = ""
    cands = JSON_OBJ_RE.findall(text)
    for s in reversed(cands):
        try:
            obj = json.loads(s)
            if isinstance(obj, dict):
                if "answer" in obj:
                    answer = str(obj["answer"]).strip()
                if "context number" in obj:
                    ctx_raw = str(obj["context number"]).strip()
                return answer, ctx_raw
        except Exception:
            continue
    m = CTX_NUM_RE.search(text)
    if m:
        ctx_raw = f"[{m.group(1)}]"
    return answer, ctx_raw

def normalize_text(s: str) -> str:
    s = unicodedata.normalize("NFKC", str(s)).strip()
    return re.sub(r"\s+", " ", s)

def to_ctx_index(ctx_raw: str) -> int:
    m = CTX_NUM_RE.search(ctx_raw)
    return int(m.group(1)) if m else 0

# -----------------------------
# 데이터 로드/포맷
# -----------------------------
def load_test(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def format_data_for_eval(raw_data: List[Dict[str, Any]]):
    """
    positive_index의 단일 문맥만 프롬프트에 포함.
    contexts 항목은 이미 "[i] ..." 형태이므로 prefix를 추가하지 않음.
    """
    out = []
    for it in raw_data:
        q   = it.get("question")
        a   = it.get("answer")
        ctxs = it.get("contexts") or []
        pix  = it.get("positive_index")
        if not q or not a or not ctxs or pix is None:
            continue
        positive_context = ctxs[pix - 1]  # 1-based index
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "system", "content": f"다음은 참고 문맥입니다:\n{positive_context}"},
            {"role": "user",   "content": q},
        ]
        out.append({
            "question": q,
            "reference_answer": a,
            "reference_ctx_number": f"[{pix}]",
            "messages": messages
        })
    return out

# -----------------------------
# 생성 함수
# -----------------------------
@torch.no_grad()
def generate_answer(model, tokenizer, messages) -> str:
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# -----------------------------
# 임베딩 & 코사인 유사도
# -----------------------------
class Embedder:
    def __init__(self, model_name: str):
        self.tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        self.model = AutoModel.from_pretrained(model_name, torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
        device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(device)
        self.device = device

    @torch.no_grad()
    def encode(self, texts: List[str], batch_size: int = 64) -> torch.Tensor:
        embs = []
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            enc = self.tok(batch, padding=True, truncation=True, return_tensors="pt").to(self.device)
            out = self.model(**enc)
            # Mean Pooling
            last_hidden = out.last_hidden_state  # (B, T, H)
            mask = enc.attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
            summed = (last_hidden * mask).sum(dim=1)
            counts = mask.sum(dim=1).clamp(min=1e-9)
            mean_pooled = summed / counts
            # L2 normalize
            mean_pooled = F.normalize(mean_pooled, p=2, dim=1)
            embs.append(mean_pooled.detach().cpu())
        return torch.cat(embs, dim=0)

    @staticmethod
    @torch.no_grad()
    def cosine_sim(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        # a, b: (N, D) L2-normalized
        return (a * b).sum(dim=1)

# -----------------------------
# 평가 루틴
# -----------------------------
def evaluate_model(name: str, model, tok, data, embedder: Embedder) -> pd.DataFrame:
    preds_ans, preds_ctx, refs_ans, refs_ctx, raws, qs = [], [], [], [], [], []
    for item in tqdm(data, desc=f"{name} 평가 중"):
        out_text = generate_answer(model, tok, item["messages"])
        ans_text, ctx_raw = safe_json_answer(out_text)
        preds_ans.append(normalize_text(ans_text))
        preds_ctx.append(ctx_raw)
        refs_ans.append(normalize_text(item["reference_answer"]))
        refs_ctx.append(item["reference_ctx_number"])
        raws.append(out_text)
        qs.append(item["question"])

    # 코사인 유사도 계산 (배치 임베딩)
    all_texts = refs_ans + preds_ans
    embs = embedder.encode(all_texts, batch_size=64)  # (2N, D)
    n = len(refs_ans)
    ref_emb = embs[:n]
    pred_emb = embs[n:]
    cos = Embedder.cosine_sim(ref_emb, pred_emb).numpy().tolist()

    # CtxAcc (context number 정확도)
    ctx_acc = [to_ctx_index(p) == to_ctx_index(r) for p, r in zip(preds_ctx, refs_ctx)]

    df = pd.DataFrame({
        "Question": qs,
        "Reference Answer": refs_ans,
        "Reference_CtxNumber": refs_ctx,

        f"{name}_Answer": preds_ans,
        f"{name}_ContextNumber": preds_ctx,
        f"{name}_Answer_Raw": raws,

        f"{name}_CosineSim": cos,
        f"{name}_CtxAcc": ctx_acc,
    })
    return df

# -----------------------------
# 메인
# -----------------------------
def main():
    # 1) 데이터 준비
    raw_test  = load_test(TEST_DATA_PATH)
    test_data = format_data_for_eval(raw_test)
    print(f"✅ 테스트 샘플: {len(test_data)}")

    # 2) 공용 토크나이저
    tok = AutoTokenizer.from_pretrained(BASE_MODEL_PATH, trust_remote_code=True, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    # 3) 평가용 임베더 로드
    embedder = Embedder(EMB_MODEL_NAME)

    # 4) Base 모델
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_PATH, trust_remote_code=True, device_map="auto",
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
    )
    base_df = evaluate_model("Base", base_model, tok, test_data, embedder)
    del base_model; torch.cuda.empty_cache()

    # 5) Finetuned (LoRA merge)
    base_for_ft = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_PATH, trust_remote_code=True, device_map="auto",
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32
    )
    ft_model = PeftModel.from_pretrained(base_for_ft, ADAPTER_PATH).merge_and_unload()
    ft_df = evaluate_model("Finetuned", ft_model, tok, test_data, embedder)
    del base_for_ft, ft_model; torch.cuda.empty_cache()

    # 6) 병합 & 저장
    df = pd.merge(
        base_df, ft_df,
        on=["Question", "Reference Answer", "Reference_CtxNumber"],
        suffixes=("_Base", "_Finetuned")
    )

    df.to_csv(OUTPUT_CSV_PATH, index=False, encoding="utf-8-sig")
    print(f"\n✅ 결과 CSV 저장 완료: {OUTPUT_CSV_PATH}")

    # 7) 콘솔 요약(평균)
    # CosineSim은 단일 지표이므로 평균만 출력, CtxAcc는 비율로 표기
    base_cos = pd.to_numeric(df["Base_CosineSim"], errors="coerce").mean()
    ft_cos   = pd.to_numeric(df["Finetuned_CosineSim"], errors="coerce").mean()
    base_ctx = pd.to_numeric(df["Base_CtxAcc"], errors="coerce").mean()
    ft_ctx   = pd.to_numeric(df["Finetuned_CtxAcc"], errors="coerce").mean()

    print("\n--- 📊 전체 평균 비교 ---")
    print(f"CosineSim | Base {base_cos:.4f} → Finetuned {ft_cos:.4f} | Δ {ft_cos - base_cos:+.4f}")
    print(f"CtxAcc    | Base {base_ctx:.4f} → Finetuned {ft_ctx:.4f} | Δ {ft_ctx - base_ctx:+.4f}")

if __name__ == "__main__":
    main()


✅ 테스트 샘플: 289


tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

adapter_config.json:   0%|          | 0.00/949 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/168M [00:00<?, ?B/s]

Finetuned 평가 중: 100%|██████████| 289/289 [05:27<00:00,  1.13s/it]



✅ 결과 CSV 저장 완료: ./quantitative_evaluation_kanana03.csv

--- 📊 전체 평균 비교 ---
CosineSim | Base 0.9004 → Finetuned 0.9526 | Δ +0.0523
CtxAcc    | Base 0.8097 → Finetuned 0.9965 | Δ +0.1869
